## specialization

This notebook introduces `:>` specialization; after running it you can declare that one part type is a kind of another.

The previous notebook defined `HeatingSystem` and `ControlSystem` as standalone types. This notebook makes them specializations of `ToastingSystem`, establishing that both are toasting-system components. The model now has a three-level type hierarchy: abstract concept → specialized type → (composition to follow in the next notebook).

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch01-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch01-cumulative.sysml` file declares the four foundational part definitions: `ToastingSystem` (abstract system concept with a doc annotation), `Heater` (with a `power : Real` attribute), `HeatingSystem` and `ControlSystem` (specializations via `:>`), and `Toaster` (composed from `heating` and `control` parts). These constructs form the structural skeleton that every later chapter extends.

In [ ]:
# Negative control: the supertype must exist in the same package or be imported.
# Specializing an undefined type raises "unresolved reference".
bad_source = """
package Bad {
    part def HeatingSystem :> UndefinedBase;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
hs = model.find("ToasterDemo::HeatingSystem")
assert hs is not None
specs = hs.specializations
print(f"HeatingSystem specializations ({len(specs)}):")
for s in specs:
    print(f"  {s.kind}: {s.declared} -> {s.target_id}")

cs = model.find("ToasterDemo::ControlSystem")
print()
print(f"ControlSystem specializes: {cs.specializations[0].target_id}")
conn.close()

`part def HeatingSystem :> ToastingSystem` is the A-F specialization; OpenSysML resolves the supertype reference and records the relationship (O-S); `hs.specializations` returns the target id (E).

Try the chapter exercise in `exercises/ch01/exercise.ipynb`: define a `HeatExchanger` that specializes `BrewUnit` and confirm the specialization records correctly.